[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_02/SFM_ch2_asymmetric_distributions/SFM_ch2_asymmetric_distributions.ipynb)

# SFM_ch2_asymmetric_distributions

Asymmetric Distributions: Skewed Student-t and GED
Description:
- Skewed Student-t distribution (Hansen, 1994)
- Generalized Error Distribution (GED / generalized normal)
- MLE fitting to S&P 500 returns
- Comparison with symmetric Student-t and Normal
- Model selection via AIC/BIC
Statistics of Financial Markets course

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats
from scipy.special import gamma, beta
from scipy.optimize import minimize
import os
import warnings
warnings.filterwarnings('ignore')

# !pip install arch

In [ ]:
# Chart style settings - Nature journal quality
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 8
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['lines.linewidth'] = 0.75

# Color palette
MAIN_BLUE = '#1A3A6E'
CRIMSON = '#DC3545'
FOREST = '#2E7D32'
AMBER = '#B5853F'
ORANGE = '#E67E22'

# Output directory
CHART_DIR = os.path.join('..', '..', '..', 'charts')
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(name):
    """Save figure with transparent background."""
    plt.savefig(os.path.join(CHART_DIR, f'{name}.pdf'),
                bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(CHART_DIR, f'{name}.png'),
                bbox_inches='tight', transparent=True, dpi=300)
    plt.close()
    print(f"   Saved: {name}.pdf/.png")

## Download Data

In [ ]:
data = yf.download('^GSPC', start='2000-01-01', end='2025-01-01', progress=False)
close = data['Close'].squeeze()
log_ret = np.log(close / close.shift(1)).dropna()

print(f"   Period: {close.index[0].strftime('%Y-%m-%d')} to {close.index[-1].strftime('%Y-%m-%d')}")
print(f"   Observations: {len(log_ret)}")
print(f"   Mean:     {log_ret.mean():.6f}")
print(f"   Std:      {log_ret.std():.6f}")
print(f"   Skewness: {log_ret.skew():.4f}")
print(f"   Kurtosis: {log_ret.kurtosis():.4f}")

## Generalized Error Distribution (GED)

In [ ]:
# GED is also known as the generalized normal distribution
# scipy.stats.gennorm with shape parameter beta:
#   beta = 1 -> Laplace (double exponential)
#   beta = 2 -> Normal
#   beta > 2 -> lighter tails than Normal
#   beta < 2 -> heavier tails than Normal

fig, ax = plt.subplots(figsize=(7, 3))

x = np.linspace(-5, 5, 500)

shapes = [
    (1.0, 'Laplace ($\\beta=1$)', CRIMSON),
    (1.5, 'GED ($\\beta=1.5$)', AMBER),
    (2.0, 'Normal ($\\beta=2$)', MAIN_BLUE),
    (3.0, 'GED ($\\beta=3$)', FOREST),
]

for shape, label, color in shapes:
    pdf = stats.gennorm.pdf(x, beta=shape)
    ax.plot(x, pdf, color=color, linewidth=1.2, label=label)

ax.set_xlabel('$x$')
ax.set_ylabel('Density')
ax.set_xlim(-5, 5)
ax.set_ylim(0, None)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=4,
          frameon=False, fontsize=7)

plt.tight_layout()
save_fig('ch2_ged_pdfs')

## GED Fit to S&P 500

In [ ]:
# Fit GED (generalized normal) to S&P 500 log-returns
ged_params = stats.gennorm.fit(log_ret)
ged_beta, ged_loc, ged_scale = ged_params

# Fit Normal for comparison
norm_loc, norm_scale = stats.norm.fit(log_ret)

# Log-likelihoods
n = len(log_ret)
ll_ged = np.sum(stats.gennorm.logpdf(log_ret, *ged_params))
ll_norm = np.sum(stats.norm.logpdf(log_ret, norm_loc, norm_scale))

# AIC / BIC  (k = number of parameters)
k_ged = 3   # beta, loc, scale
k_norm = 2  # loc, scale

aic_ged  = -2 * ll_ged  + 2 * k_ged
bic_ged  = -2 * ll_ged  + np.log(n) * k_ged
aic_norm = -2 * ll_norm + 2 * k_norm
bic_norm = -2 * ll_norm + np.log(n) * k_norm

print(f"   GED fit:    beta={ged_beta:.4f}, loc={ged_loc:.6f}, scale={ged_scale:.6f}")
print(f"   Normal fit: loc={norm_loc:.6f}, scale={norm_scale:.6f}")
print(f"")
print(f"   GED    -> LL={ll_ged:.2f},  AIC={aic_ged:.2f},  BIC={bic_ged:.2f}")
print(f"   Normal -> LL={ll_norm:.2f}, AIC={aic_norm:.2f}, BIC={bic_norm:.2f}")

# Plot histogram + fitted densities on log scale
fig, ax = plt.subplots(figsize=(7, 3))

x = np.linspace(log_ret.min() * 1.1, log_ret.max() * 1.1, 500)

ax.hist(log_ret, bins=150, density=True, alpha=0.3, color='gray',
        edgecolor='none', label='S&P 500 returns')

ax.plot(x, stats.gennorm.pdf(x, *ged_params), color=CRIMSON, linewidth=1.2,
        label=f'GED ($\\beta$={ged_beta:.2f})')
ax.plot(x, stats.norm.pdf(x, norm_loc, norm_scale), color=MAIN_BLUE,
        linewidth=1.2, linestyle='--', label='Normal')

ax.set_yscale('log')
ax.set_xlabel('Log-return')
ax.set_ylabel('Density (log scale)')
ax.set_ylim(1e-2, None)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3,
          frameon=False, fontsize=7)

plt.tight_layout()
save_fig('ch2_ged_fit')

## Skewed Student-t Distribution

In [ ]:
# Hansen (1994) skewed Student-t distribution
# Parameters: eta (degrees of freedom, eta > 2), lam (skewness, -1 < lam < 1)
#
# The density is:
#   f(z | eta, lam) = { b*c*(1 + 1/(eta-2)*((b*z+a)/(1-lam))^2)^(-(eta+1)/2)  if z < -a/b
#                     { b*c*(1 + 1/(eta-2)*((b*z+a)/(1+lam))^2)^(-(eta+1)/2)  if z >= -a/b
#   where:
#     c = gamma((eta+1)/2) / (sqrt(pi*(eta-2)) * gamma(eta/2))
#     a = 4*lam*c*(eta-2)/(eta-1)
#     b = sqrt(1 + 3*lam^2 - a^2)

def skewt_pdf(x, eta, lam):
    """Hansen (1994) skewed Student-t PDF.
    
    Parameters
    ----------
    x : array-like
        Evaluation points (standardized).
    eta : float
        Degrees of freedom (eta > 2).
    lam : float
        Skewness parameter (-1 < lam < 1).
    """
    c = gamma((eta + 1) / 2) / (np.sqrt(np.pi * (eta - 2)) * gamma(eta / 2))
    a = 4 * lam * c * (eta - 2) / (eta - 1)
    b = np.sqrt(1 + 3 * lam**2 - a**2)
    
    y = b * np.asarray(x) + a
    
    pdf = np.where(
        x < -a / b,
        b * c * (1 + (1 / (eta - 2)) * (y / (1 - lam))**2)**(-(eta + 1) / 2),
        b * c * (1 + (1 / (eta - 2)) * (y / (1 + lam))**2)**(-(eta + 1) / 2)
    )
    return pdf


def skewt_logpdf(x, eta, lam):
    """Log-density of Hansen's skewed-t."""
    return np.log(skewt_pdf(x, eta, lam) + 1e-300)


# Plot skewed-t PDFs for different skewness parameters
fig, ax = plt.subplots(figsize=(7, 3))

x = np.linspace(-5, 5, 500)
eta = 5  # degrees of freedom

configs = [
    (-0.3, 'Left-skewed ($\\lambda=-0.3$)', CRIMSON),
    (0.0,  'Symmetric ($\\lambda=0$)', MAIN_BLUE),
    (0.3,  'Right-skewed ($\\lambda=0.3$)', FOREST),
]

for lam, label, color in configs:
    pdf = skewt_pdf(x, eta, lam)
    ax.plot(x, pdf, color=color, linewidth=1.2, label=label)

ax.set_xlabel('$x$')
ax.set_ylabel('Density')
ax.set_xlim(-5, 5)
ax.set_ylim(0, None)
ax.set_title(f'Hansen Skewed Student-t ($\\eta={eta}$)', fontweight='bold', fontsize=9)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3,
          frameon=False, fontsize=7)

plt.tight_layout()
save_fig('ch2_skewed_t_pdfs')

## Model Comparison

In [ ]:
# --- Fit Normal ---
norm_mu, norm_sigma = stats.norm.fit(log_ret)
ll_normal = np.sum(stats.norm.logpdf(log_ret, norm_mu, norm_sigma))
k_normal = 2

# --- Fit Student-t ---
t_df, t_loc, t_scale = stats.t.fit(log_ret)
ll_t = np.sum(stats.t.logpdf(log_ret, t_df, t_loc, t_scale))
k_t = 3

# --- Fit GED (generalized normal) ---
ged_beta_, ged_loc_, ged_scale_ = stats.gennorm.fit(log_ret)
ll_ged_ = np.sum(stats.gennorm.logpdf(log_ret, ged_beta_, ged_loc_, ged_scale_))
k_ged_ = 3

# --- Fit Skewed Student-t (Hansen) ---
# Standardize returns for skewed-t fitting
ret_std = (log_ret - log_ret.mean()) / log_ret.std()

def neg_ll_skewt(params):
    eta, lam = params
    if eta <= 2.01 or lam <= -0.99 or lam >= 0.99:
        return 1e10
    return -np.sum(skewt_logpdf(ret_std, eta, lam))

# Grid search for good starting values
best_nll = np.inf
best_x0 = [5, 0]
for eta0 in [4, 6, 10]:
    for lam0 in [-0.2, 0, 0.2]:
        nll = neg_ll_skewt([eta0, lam0])
        if nll < best_nll:
            best_nll = nll
            best_x0 = [eta0, lam0]

res = minimize(neg_ll_skewt, best_x0, method='Nelder-Mead',
               options={'maxiter': 5000, 'xatol': 1e-8, 'fatol': 1e-8})
eta_hat, lam_hat = res.x

# Reconstruct log-likelihood on original scale:
# log f(r) = log f_std((r-mu)/sigma) - log(sigma)
mu_r = log_ret.mean()
sigma_r = log_ret.std()
ll_skewt = -res.fun - n * np.log(sigma_r)
k_skewt = 4  # mu, sigma, eta, lam

# --- Compute AIC and BIC ---
n = len(log_ret)
results = []
for name, ll, k in [
    ('Normal',     ll_normal, k_normal),
    ('Student-t',  ll_t,     k_t),
    ('GED',        ll_ged_,  k_ged_),
    ('Skewed-t',   ll_skewt, k_skewt),
]:
    aic = -2 * ll + 2 * k
    bic = -2 * ll + np.log(n) * k
    results.append({'Model': name, 'k': k, 'LogLik': ll,
                    'AIC': aic, 'BIC': bic})

df_results = pd.DataFrame(results).set_index('Model')
print("   Model comparison (S&P 500 log-returns)")
print("   " + "=" * 55)
print(df_results.to_string())
print()
print(f"   Skewed-t fit: eta={eta_hat:.2f} (df), lambda={lam_hat:.4f} (skew)")
print(f"   GED fit:      beta={ged_beta_:.4f}")
print(f"   Student-t:    df={t_df:.2f}")

# --- Bar chart ---
fig, axes = plt.subplots(1, 2, figsize=(8, 3))

models = df_results.index.tolist()
colors = [MAIN_BLUE, CRIMSON, AMBER, FOREST]
x_pos = np.arange(len(models))

# AIC panel
axes[0].bar(x_pos, df_results['AIC'].values, color=colors, alpha=0.85, width=0.6)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(models, fontsize=7)
axes[0].set_ylabel('AIC')
axes[0].set_title('AIC (lower is better)', fontweight='bold', fontsize=9)

# BIC panel
axes[1].bar(x_pos, df_results['BIC'].values, color=colors, alpha=0.85, width=0.6)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(models, fontsize=7)
axes[1].set_ylabel('BIC')
axes[1].set_title('BIC (lower is better)', fontweight='bold', fontsize=9)

# Remove top/right spines on second panel
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
save_fig('ch2_asymmetric_comparison')